In [ ]:
# ==========================================
# 3_evaluation.ipynb
# ==========================================

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
import numpy as np
import os
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from notebook1_qcnn_definitions import StudentModel, TeacherModel, HierarchicalClassifier

# -----------------------------
# Test Dataset Loader
# -----------------------------
class KneeOATestDataset(Dataset):    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        for cls in self.classes:
            cls_path = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_path):
                continue
            for f in os.listdir(cls_path):
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((os.path.join(cls_path, f), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# -----------------------------
# Config
# -----------------------------
test_root = '/PATH/TO/TEST/DATA'
batch_size = 8
device = 'cuda' if torch.cuda.is_available() else 'cpu'

transform_test = T.Compose([
    T.Resize((32, 32)),
    T.ToTensor()
])

test_dataset = KneeOATestDataset(test_root, transform=transform_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# -----------------------------
# Load Models
# -----------------------------
student_model = StudentModel(n_qubits=8, unitary_name='U_SU4').to(device)
teacher_model = TeacherModel(num_classes=len(test_dataset.classes)).to(device)
hierarchical_classifier = HierarchicalClassifier(input_dim=8, num_fine_classes=len(test_dataset.classes)).to(device)

student_model.load_state_dict(torch.load('stage3_outputs/student_model.pt', map_location=device))
teacher_model.load_state_dict(torch.load('stage3_outputs/teacher_model.pt', map_location=device))
hierarchical_classifier.load_state_dict(torch.load('stage3_outputs/hier_classifier.pt', map_location=device))

student_model.eval()
teacher_model.eval()
hierarchical_classifier.eval()

# -----------------------------
# Evaluation Loop
# -----------------------------
y_true, y_pred = [], []

with torch.no_grad():
    for data, labels in test_loader:
        data = data.to(device)
        labels = labels.to(device)

        features = student_model(data)
        _, fine_logits = hierarchical_classifier(features)
        preds = fine_logits.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# -----------------------------
# Sample Predictions Display
# -----------------------------
def imshow(img, title=None):
    img = img.numpy().transpose((1, 2, 0))
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis('off')

# plot first batch
data_iter = iter(test_loader)
images, labels = next(data_iter)
with torch.no_grad():
    preds = hierarchical_classifier(student_model(images.to(device)))[1].argmax(dim=1)

plt.figure(figsize=(12, 6))
for idx in range(min(8, len(images))):
    plt.subplot(2, 4, idx+1)
    imshow(images[idx])
    plt.title(f"T:{test_dataset.classes[labels[idx]]}\nP:{test_dataset.classes[preds[idx]]}")
plt.show()